In [9]:
import sys
import subprocess
import importlib.util
import re
import json
from pathlib import Path
from datetime import datetime

# ============================================================
# CHECK AND INSTALL REQUIRED PDF VERIFICATION PACKAGE
# ============================================================

if importlib.util.find_spec("pypdf") is None:
    print("Installing missing PDF verification package...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "pypdf"
    ])

from pypdf import PdfReader
from docx import Document
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph as PDFParagraph,
    Spacer, Table as PDFTable, TableStyle
)

from docx.oxml.table import CT_Tbl
from docx.oxml.text.paragraph import CT_P
from docx.table import Table as DocxTable
from docx.text.paragraph import Paragraph

# ============================================================
# PROJECT FILE CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\hp\Analysis in Global Supply Chain Operations"
)

DOCX_FILE = (
    PROJECT_ROOT / "government_executive_summary"
    / "Government_Executive_Summary.docx"
)

PDF_FILE = (
    PROJECT_ROOT / "government_executive_summary"
    / "Government_Executive_Summary.pdf"
)

REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

VERIFICATION_FILE = (
    REPORT_DIR / "government_executive_summary_verification.json"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 90)
print("GOVERNMENT EXECUTIVE SUMMARY PDF REGENERATION AND VERIFICATION")
print("=" * 90)

if not DOCX_FILE.exists():
    raise FileNotFoundError(f"DOCX not found: {DOCX_FILE}")

# ============================================================
# LOAD SOURCE DOCX
# ============================================================

doc = Document(DOCX_FILE)

if not doc.paragraphs and not doc.tables:
    raise ValueError("The source DOCX contains no readable content.")

print("\nSOURCE DOCUMENT")
print("-" * 90)
print(f"DOCX paragraphs: {len(doc.paragraphs)}")
print(f"DOCX tables: {len(doc.tables)}")

# ============================================================
# PREPARE PDF STYLES
# ============================================================

styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    "GovernmentTitle",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=18,
    leading=23,
    alignment=TA_CENTER,
    textColor=colors.HexColor("#1F4E79"),
    spaceAfter=12
)

heading_style = ParagraphStyle(
    "GovernmentHeading",
    parent=styles["Heading1"],
    fontName="Helvetica-Bold",
    fontSize=12,
    leading=15,
    textColor=colors.HexColor("#1F4E79"),
    spaceBefore=11,
    spaceAfter=6,
    keepWithNext=True
)

subheading_style = ParagraphStyle(
    "GovernmentSubheading",
    parent=styles["Heading2"],
    fontName="Helvetica-Bold",
    fontSize=10,
    leading=13,
    textColor=colors.HexColor("#365F91"),
    spaceBefore=8,
    spaceAfter=4
)

body_style = ParagraphStyle(
    "GovernmentBody",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=8.5,
    leading=12,
    spaceAfter=6
)

bullet_style = ParagraphStyle(
    "GovernmentBullet",
    parent=body_style,
    leftIndent=13,
    firstLineIndent=-8,
    spaceAfter=4
)

cell_style = ParagraphStyle(
    "GovernmentCell",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=6.5,
    leading=8
)

header_cell_style = ParagraphStyle(
    "GovernmentHeaderCell",
    parent=cell_style,
    fontName="Helvetica-Bold",
    textColor=colors.white
)

def escape_pdf_text(value):
    value = str(value)
    return (
        value.replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace("\n", "<br/>")
    )

# ============================================================
# CONVERT DOCX CONTENT INTO PDF CONTENT
# ============================================================

story = []

for element in doc.element.body.iterchildren():

    if isinstance(element, CT_P):
        paragraph = Paragraph(element, doc)
        text_value = paragraph.text.strip()

        if not text_value:
            story.append(Spacer(1, 3))
            continue

        style_name = (
            paragraph.style.name
            if paragraph.style else ""
        )

        if style_name == "Title":
            selected_style = title_style

        elif style_name.startswith("Heading 1"):
            selected_style = heading_style

        elif style_name.startswith("Heading 2"):
            selected_style = subheading_style

        elif "List Bullet" in style_name:
            selected_style = bullet_style
            text_value = "&#8226; " + escape_pdf_text(text_value)

        else:
            selected_style = body_style

        if selected_style != bullet_style:
            text_value = escape_pdf_text(text_value)

        story.append(
            PDFParagraph(text_value, selected_style)
        )

    elif isinstance(element, CT_Tbl):
        table = DocxTable(element, doc)
        table_data = []

        for row_index, row in enumerate(table.rows):
            output_row = []

            for cell in row.cells:
                selected_style = (
                    header_cell_style
                    if row_index == 0
                    else cell_style
                )

                output_row.append(
                    PDFParagraph(
                        escape_pdf_text(cell.text.strip()),
                        selected_style
                    )
                )

            table_data.append(output_row)

        if table_data:
            column_count = max(
                len(row) for row in table_data
            )

            available_width = A4[0] - 1.2 * inch
            column_width = available_width / column_count

            pdf_table = PDFTable(
                table_data,
                colWidths=[column_width] * column_count,
                repeatRows=1
            )

            pdf_table.setStyle(TableStyle([
                (
                    "BACKGROUND",
                    (0, 0),
                    (-1, 0),
                    colors.HexColor("#1F4E79")
                ),
                ("GRID", (0, 0), (-1, -1), 0.35, colors.grey),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                (
                    "ROWBACKGROUNDS",
                    (0, 1),
                    (-1, -1),
                    [colors.white, colors.HexColor("#F2F6FA")]
                ),
                ("LEFTPADDING", (0, 0), (-1, -1), 4),
                ("RIGHTPADDING", (0, 0), (-1, -1), 4),
                ("TOPPADDING", (0, 0), (-1, -1), 4),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 4)
            ]))

            story.append(Spacer(1, 5))
            story.append(pdf_table)
            story.append(Spacer(1, 8))

# ============================================================
# EXPORT REGENERATED PDF
# ============================================================

def add_footer(canvas, pdf_document):
    canvas.saveState()
    canvas.setFont("Helvetica", 7)
    canvas.setFillColor(colors.grey)

    canvas.drawCentredString(
        A4[0] / 2,
        0.35 * inch,
        f"Government Executive Summary | Page {pdf_document.page}"
    )

    canvas.restoreState()

pdf_document = SimpleDocTemplate(
    str(PDF_FILE),
    pagesize=A4,
    rightMargin=0.6 * inch,
    leftMargin=0.6 * inch,
    topMargin=0.6 * inch,
    bottomMargin=0.65 * inch,
    title="Government Executive Summary - Global Supply Chain Operations",
    author="Analysis in Global Supply Chain Operations"
)

pdf_document.build(
    story,
    onFirstPage=add_footer,
    onLaterPages=add_footer
)

# ============================================================
# VERIFY DOCX AND PDF CONTENT
# ============================================================

reopened_docx = Document(DOCX_FILE)

docx_text = "\n".join(
    paragraph.text
    for paragraph in reopened_docx.paragraphs
)

for table in reopened_docx.tables:
    for row in table.rows:
        docx_text += "\n" + " ".join(
            cell.text for cell in row.cells
        )

reader = PdfReader(str(PDF_FILE))

pdf_text = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

def normalize_text(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower()
    )

normalized_docx = normalize_text(docx_text)
normalized_pdf = normalize_text(pdf_text)

required_sections = [
    "Executive Overview",
    "Supply Chain Problem",
    "Dataset Overview",
    "Delivery Performance Findings",
    "Delay Risk Findings",
    "Shipping Mode Findings",
    "Regional and Market Findings",
    "Customer Segment Impact",
    "Key Performance Indicators",
    "Identified Operational Challenges",
    "Recommended Interventions",
    "Expected Operational Benefits",
    "Data Limitations",
    "Conclusion",
    "References"
]

docx_sections_pass = all(
    normalize_text(section) in normalized_docx
    for section in required_sections
)

pdf_sections_pass = all(
    normalize_text(section) in normalized_pdf
    for section in required_sections
)

docx_reopens = (
    len(reopened_docx.paragraphs) > 0
    and DOCX_FILE.stat().st_size > 0
)

pdf_reopens = (
    len(reader.pages) > 0
    and PDF_FILE.stat().st_size > 0
)

pdf_text_available = len(normalized_pdf) > 500

pdf_text_verification = (
    pdf_reopens
    and pdf_text_available
    and pdf_sections_pass
)

# ============================================================
# SAVE VERIFICATION REPORT
# ============================================================

verification = {
    "project": "Analysis in Global Supply Chain Operations",
    "verification_timestamp": datetime.now().isoformat(),
    "docx_path": str(DOCX_FILE),
    "pdf_path": str(PDF_FILE),
    "docx_size_bytes": DOCX_FILE.stat().st_size,
    "pdf_size_bytes": PDF_FILE.stat().st_size,
    "pdf_page_count": len(reader.pages),
    "docx_paragraph_count": len(reopened_docx.paragraphs),
    "docx_table_count": len(reopened_docx.tables),
    "pdf_extracted_text_characters": len(pdf_text),
    "required_sections": required_sections,
    "validation_checks": {
        "docx_exists": DOCX_FILE.exists(),
        "docx_reopens_successfully": docx_reopens,
        "docx_required_sections_present": docx_sections_pass,
        "pdf_exists": PDF_FILE.exists(),
        "pdf_reopens_successfully": pdf_reopens,
        "pdf_text_available": pdf_text_available,
        "pdf_required_sections_present": pdf_sections_pass,
        "pdf_text_verification": pdf_text_verification
    }
}

verification["all_checks_passed"] = all(
    verification["validation_checks"].values()
)

with open(
    VERIFICATION_FILE,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        verification,
        file,
        indent=4,
        ensure_ascii=False
    )

# ============================================================
# DISPLAY FINAL RESULTS
# ============================================================

print("\nDOCUMENT VERIFICATION")
print("=" * 90)

for check, passed in verification["validation_checks"].items():
    print(
        f"{check.replace('_', ' ').title():45} "
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nPDF DETAILS")
print("-" * 90)
print(f"PDF pages: {len(reader.pages)}")
print(f"Extracted text characters: {len(pdf_text):,}")
print(
    "Required sections found: "
    f"{sum(normalize_text(s) in normalized_pdf for s in required_sections)}"
    f"/{len(required_sections)}"
)

print("\nEXPORTED DOCUMENTS")
print("-" * 90)
print(f"DOCX: {DOCX_FILE}")
print(f"PDF: {PDF_FILE}")
print(f"Verification report: {VERIFICATION_FILE}")

print("\n" + "=" * 90)

if verification["all_checks_passed"]:
    print("GOVERNMENT EXECUTIVE SUMMARY COMPLETED SUCCESSFULLY")
    print("DOCX AND PDF DOCUMENTS VERIFIED")
    print("ALL VALIDATION CHECKS PASSED")
else:
    failed_checks = [
        key
        for key, value in verification["validation_checks"].items()
        if not value
    ]

    print("DOCUMENT VERIFICATION FAILED")
    print("Failed checks:", failed_checks)

print("=" * 90)

GOVERNMENT EXECUTIVE SUMMARY PDF REGENERATION AND VERIFICATION

SOURCE DOCUMENT
------------------------------------------------------------------------------------------
DOCX paragraphs: 71
DOCX tables: 7

DOCUMENT VERIFICATION
Docx Exists                                   PASS
Docx Reopens Successfully                     PASS
Docx Required Sections Present                PASS
Pdf Exists                                    PASS
Pdf Reopens Successfully                      PASS
Pdf Text Available                            PASS
Pdf Required Sections Present                 PASS
Pdf Text Verification                         PASS

PDF DETAILS
------------------------------------------------------------------------------------------
PDF pages: 4
Extracted text characters: 10,979
Required sections found: 15/15

EXPORTED DOCUMENTS
------------------------------------------------------------------------------------------
DOCX: C:\Users\hp\Analysis in Global Supply Chain Operations\governmen